# Endpoint–Artifact Evidence Table

This notebook prepares endpoint-security telemetry for Neo4j loading.

The objective is to preserve the complete endpoint–artifact observation
dataset while separately recording explicit malicious security evidence.

An artifact is identified by its SHA-256 hash.

The resulting data model distinguishes:

- endpoint activity and artifact observations
- explicit `FileType: Malicious` security detections
- investigation-oriented activity metrics

Malicious classification is therefore stored as relationship-level evidence
rather than being used to remove non-malicious endpoint observations.

In [14]:
# Import pathlib for portable project-relative file paths.
from pathlib import Path

# Import pandas for data loading, cleaning, aggregation, and CSV export.
import pandas as pd

# Import display for readable notebook tables.
from IPython.display import display


# Resolve the input and output locations relative to this notebook.
DATA_DIR = Path("../data")

SOURCE_PATH = DATA_DIR / "endpoint_security.csv"
ARTIFACT_OUTPUT_PATH = DATA_DIR / "artifact_level_table.csv"
EDGE_OUTPUT_PATH = DATA_DIR / "endpoint_artifact_edges.csv"

print("Source:", SOURCE_PATH.resolve())
print("Artifact output:", ARTIFACT_OUTPUT_PATH.resolve())
print("Edge output:", EDGE_OUTPUT_PATH.resolve())

Source: C:\Users\Asus\Desktop\web_activity\data\endpoint_security.csv
Artifact output: C:\Users\Asus\Desktop\web_activity\data\artifact_level_table.csv
Edge output: C:\Users\Asus\Desktop\web_activity\data\endpoint_artifact_edges.csv


In [15]:
# Load the complete endpoint-security telemetry.
# Unlike the previous version, no malicious-only filter is applied here.
df = pd.read_csv(SOURCE_PATH)

print("Raw rows:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Raw rows: 23540
Columns: ['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message']


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message
0,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,No suspicious file found by type or extension.
1,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,File: 53EDA42869D426262B752F088EA605187284ACE8...
2,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No suspicious file found by type or extension.
3,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No Malicious files found.
4,EPFFTO5T,host-508b,FA:2F:46:B7:14:23,unit-delta,quarantine,01-09-2026 08:50:38,01-09-2026 08:50:38,No suspicious file found by type or extension.


## 1. Select the artifact observations

The source message contains the file name and SHA-256. We keep only:

- endpoints in the defined analysis scope
- messages containing `FileType: Malicious`
- valid 64-character SHA-256 values


In [16]:
# Validate the core columns required for endpoint-artifact correlation.
required_columns = {
    "endpoint_id",
    "event_time",
    "message",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing from the source dataset: "
        f"{sorted(missing_columns)}"
    )

print("Required source columns are present.")

Required source columns are present.


## Parse artifact identity from security telemetry

The source security message contains the file name and SHA-256 value.

SHA-256 is used as the artifact identity because the same artifact may be
associated with multiple filenames or observations.

In [17]:
# Convert the security message to a safe string representation before
# applying regular expressions.
message = df["message"].fillna("").astype(str)

# Extract the filename from the structured security message.
df["file_name"] = message.str.extract(
    r"^File:\s*(.*?)\s*\|",
    expand=False,
)

# Extract the SHA-256 artifact identifier.
df["sha256"] = message.str.extract(
    r"SHA256:\s*([0-9a-fA-F]{64})",
    expand=False,
)

# Normalize SHA-256 values so equivalent hashes have identical representation.
df["sha256"] = df["sha256"].str.strip().str.lower()

# Normalize filenames for consistent aggregation.
df["file_name"] = df["file_name"].str.strip()

# Parse the source event timestamp using the format present in the dataset.
df["event_time"] = pd.to_datetime(
    df["event_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce",
)

print(
    "Rows containing a valid SHA-256:",
    df["sha256"].notna().sum(),
)

print(
    "Unique SHA-256 artifacts:",
    df["sha256"].dropna().nunique(),
)

Rows containing a valid SHA-256: 157
Unique SHA-256 artifacts: 60


## 2. Build the artifact-level table

One row = one unique SHA-256 artifact.

Important: filenames are aggregated rather than selecting the first filename. This preserves cases where the same SHA-256 appears under multiple filenames.

In [18]:
# Keep observations that contain the minimum information required to create
# an endpoint-artifact relationship.
observation_scope = df[
    df["endpoint_id"].notna()
    & df["sha256"].notna()
].copy()

print("Endpoint-artifact observations:", len(observation_scope))
print(
    "Unique endpoints:",
    observation_scope["endpoint_id"].nunique(),
)
print(
    "Unique artifacts:",
    observation_scope["sha256"].nunique(),
)

Endpoint-artifact observations: 157
Unique endpoints: 38
Unique artifacts: 60


## Explicit malicious security evidence

The raw security telemetry explicitly identifies malicious files using:

`FileType: Malicious`

This is treated as direct security evidence.

Importantly, benign observations are not removed from the dataset.

Instead, malicious evidence is correlated to the corresponding
`endpoint_id + sha256` relationship.

In [19]:
# Identify the subset of raw telemetry that explicitly reports a malicious file.
malicious = observation_scope[
    observation_scope["message"]
    .fillna("")
    .str.contains(
        "FileType: Malicious",
        case=False,
        na=False,
    )
].copy()

print(
    "Explicit malicious records:",
    len(malicious),
)

print(
    "Endpoints with explicit malicious detections:",
    malicious["endpoint_id"].nunique(),
)

print(
    "Artifacts with explicit malicious detections:",
    malicious["sha256"].nunique(),
)

Explicit malicious records: 134
Endpoints with explicit malicious detections: 33
Artifacts with explicit malicious detections: 51


assert scope['endpoint_id'].nunique() == 22
assert len(scope) == 87
assert artifact_level['sha256'].nunique() == 49
assert artifact_level['record_count'].sum() == 87
assert endpoint_file_level['record_count'].sum() == 87
assert (artifact_level['file_name_count'] >= 1).all()

print('Validation passed.')
print('  Observation rows :', len(scope))
print('  Endpoints        :', scope['endpoint_id'].nunique())
print('  Unique artifacts :', artifact_level['sha256'].nunique())
print('  Total records    :', artifact_level['record_count'].sum())
print('\nRecord-count distribution:')
print(artifact_level['record_count'].value_counts().sort_index())


In [20]:
# Extract quarantine timestamps from malicious security messages when present.
malicious["quarantined_at"] = malicious["message"].str.extract(
    r"Quarantined At:\s*([^|]+)",
    expand=False,
)

# Parse quarantine timestamps into pandas datetime values.
malicious["quarantined_at"] = pd.to_datetime(
    malicious["quarantined_at"],
    dayfirst=True,
    errors="coerce",
)

# Remove incomplete endpoint-artifact pairs because they cannot be correlated
# reliably to a graph relationship.
malicious = malicious.dropna(
    subset=["endpoint_id", "sha256"]
).copy()

display(
    malicious[
        [
            "endpoint_id",
            "sha256",
            "file_name",
            "event_time",
            "quarantined_at",
        ]
    ].head()
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_1660\2611093385.py:8: UserWarning: Parsing dates in %Y-%m-%d %H:%M  format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  malicious["quarantined_at"] = pd.to_datetime(


,endpoint_id,sha256,file_name,event_time,quarantined_at
1,EPZRAPEL,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,53EDA42869D426262B752F088EA605187284ACE8,2026-09-01 04:58:58,2026-08-31 13:21:00
394,EPOC6Y4R,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,recently-used.xbel,2026-09-01 08:59:34,2026-08-31 13:35:00
671,EPOC6Y4R,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,recently-used.xbel,2026-09-01 18:21:26,2026-09-01 13:33:00
995,EPJYEC7F,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,recently-used.xbel,2026-09-01 08:09:22,2026-08-31 13:17:00
1045,EPMQTKSL,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,recently-used.xbel,2026-09-02 08:37:11,2026-09-01 13:56:00


artifact_level_csv = artifact_level.copy()
artifact_level_csv['file_names'] = artifact_level_csv['file_names'].apply(lambda x: ' | '.join(x))
artifact_level_csv['endpoint_ids'] = artifact_level_csv['endpoint_ids'].apply(lambda x: ' | '.join(x))

artifact_level_csv.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH.resolve()}')


## Build artifact-level table

The artifact-level table contains one row per unique SHA-256 artifact.

Artifact-level properties describe the artifact across all observations.
Endpoint-specific properties remain on the endpoint-artifact relationship.

In [21]:
# Aggregate all observations by SHA-256 artifact.
artifact_level = (
    observation_scope
    .groupby("sha256", as_index=False)
    .agg(
        file_names=(
            "file_name",
            lambda values: sorted(set(values.dropna())),
        ),
        record_count=("sha256", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max"),
        endpoint_count=("endpoint_id", "nunique"),
        endpoint_ids=(
            "endpoint_id",
            lambda values: sorted(set(values.dropna())),
        ),
    )
)

# Record how many distinct filenames are associated with each artifact.
artifact_level["file_name_count"] = (
    artifact_level["file_names"].apply(len)
)

print("Unique artifacts:", len(artifact_level))

display(artifact_level.head())

Unique artifacts: 60


,sha256,file_names,record_count,first_event,last_event,endpoint_count,endpoint_ids,file_name_count
0,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV],1
1,09ffa00c93df8c4e4780c36fcb6424d21d1ec67a84c0b1...,[webex.desktop],1,2026-09-03 09:09:31,2026-09-03 09:09:31,1,[EPGW2NY4],1
2,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV],1
3,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"[Buy CP PLUS Dashboard Camera, Warranty NONE 1...",2,2026-09-03 19:03:33,2026-09-04 18:10:01,1,[EPAGTKNF],1
4,2231ec449282f6caa46d66c6c9ecadd79b2b29a93d2dcd...,[org.gnome.Screenshot.desktop],1,2026-09-16 05:37:25,2026-09-16 05:37:25,1,[EPUJL36T],1


In [22]:
# Validate the artifact-level table before writing it to disk.
assert artifact_level["sha256"].is_unique, (
    "Each artifact-level row must have a unique SHA-256."
)

assert (
    artifact_level["record_count"].sum()
    == len(observation_scope)
), (
    "Artifact record counts must account for every valid "
    "endpoint-artifact observation."
)

print("Artifact-level validation passed.")
print("Artifacts:", len(artifact_level))
print("Observations represented:", artifact_level["record_count"].sum())

Artifact-level validation passed.
Artifacts: 60
Observations represented: 157


In [23]:
# Export the artifact-level table in a CSV-compatible representation.
artifact_export = artifact_level.copy()

# Convert Python lists into pipe-delimited strings for portable CSV storage.
artifact_export["file_names"] = artifact_export["file_names"].apply(
    lambda values: " | ".join(values)
)

artifact_export["endpoint_ids"] = artifact_export["endpoint_ids"].apply(
    lambda values: " | ".join(values)
)

artifact_export.to_csv(
    ARTIFACT_OUTPUT_PATH,
    index=False,
)

print(
    f"Saved artifact-level table: "
    f"{ARTIFACT_OUTPUT_PATH.resolve()}"
)

Saved artifact-level table: C:\Users\Asus\Desktop\web_activity\data\artifact_level_table.csv


## Build endpoint–artifact relationships

Each row represents one unique:

`endpoint_id + sha256`

pair.

Relationship properties contain endpoint-specific observation counts,
timestamps, and explicit malicious-security evidence.

This is the table consumed by the Neo4j import notebook.

In [24]:
# Aggregate the complete observation dataset to one endpoint-artifact pair.
# This preserves non-malicious observations as well as malicious observations.
endpoint_artifact_edges = (
    observation_scope
    .groupby(
        ["endpoint_id", "sha256"],
        as_index=False,
    )
    .agg(
        file_names=(
            "file_name",
            lambda values: sorted(set(values.dropna())),
        ),
        record_count=("sha256", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max"),
    )
)

# Store the number of distinct filenames associated with each pair.
endpoint_artifact_edges["file_name_count"] = (
    endpoint_artifact_edges["file_names"].apply(len)
)

print(
    "Endpoint-artifact relationships:",
    len(endpoint_artifact_edges),
)

print(
    "Endpoints:",
    endpoint_artifact_edges["endpoint_id"].nunique(),
)

print(
    "Artifacts:",
    endpoint_artifact_edges["sha256"].nunique(),
)

display(endpoint_artifact_edges.head())

Endpoint-artifact relationships: 72
Endpoints: 38
Artifacts: 60


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count
0,EP2SJ6NE,93ffd792f92437a850e4939c84442391a624e20446cea2...,[libreoffice-calc.desktop],1,2026-09-26 11:06:14,2026-09-26 11:06:14,1
1,EP3GALVS,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,[recently-used.xbel],2,2026-09-19 16:35:19,2026-09-21 09:17:02,1
2,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
3,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
4,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1


In [25]:
# Aggregate malicious telemetry separately at the same endpoint-artifact grain.
# This allows explicit security evidence to be attached to the corresponding
# OBSERVED relationship without removing ordinary observations.
malicious_edge_summary = (
    malicious
    .groupby(
        ["endpoint_id", "sha256"],
        as_index=False,
    )
    .agg(
        malicious_detected=(
            "sha256",
            lambda values: True,
        ),
        malicious_detection_count=(
            "sha256",
            "size",
        ),
        malicious_file_names=(
            "file_name",
            lambda values: sorted(set(values.dropna())),
        ),
        first_malicious_detection=(
            "event_time",
            "min",
        ),
        last_malicious_detection=(
            "event_time",
            "max",
        ),
        first_quarantine_time=(
            "quarantined_at",
            "min",
        ),
        last_quarantine_time=(
            "quarantined_at",
            "max",
        ),
    )
)

print(
    "Endpoint-artifact pairs with explicit malicious evidence:",
    len(malicious_edge_summary),
)

display(malicious_edge_summary.head())

Endpoint-artifact pairs with explicit malicious evidence: 60


,endpoint_id,sha256,malicious_detected,malicious_detection_count,malicious_file_names,first_malicious_detection,last_malicious_detection,first_quarantine_time,last_quarantine_time
0,EP2SJ6NE,93ffd792f92437a850e4939c84442391a624e20446cea2...,True,1,[libreoffice-calc.desktop],2026-09-26 11:06:14,2026-09-26 11:06:14,2026-09-25 11:34:00,2026-09-25 11:34:00
1,EP3GALVS,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,True,2,[recently-used.xbel],2026-09-19 16:35:19,2026-09-21 09:17:02,2026-09-19 13:18:00,2026-09-19 13:18:00
2,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,True,2,[load-hub-i18n.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
3,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,True,2,[shopping_iframe_driver.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
4,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,True,2,[tokenized-card.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00


In [26]:
# Left-join malicious evidence onto the complete endpoint-artifact table.
# A left join is essential: endpoint-artifact pairs without malicious
# detections must remain in the final graph.
endpoint_artifact_edges = endpoint_artifact_edges.merge(
    malicious_edge_summary,
    on=["endpoint_id", "sha256"],
    how="left",
)

# Fill missing malicious flags with False for relationships without
# explicit security detections.
endpoint_artifact_edges["malicious_detected"] = (
    endpoint_artifact_edges["malicious_detected"]
    .fillna(False)
    .astype(bool)
)

# Fill missing detection counts with zero.
endpoint_artifact_edges["malicious_detection_count"] = (
    endpoint_artifact_edges["malicious_detection_count"]
    .fillna(0)
    .astype(int)
)

# Replace missing filename lists with empty lists.
endpoint_artifact_edges["malicious_file_names"] = (
    endpoint_artifact_edges["malicious_file_names"]
    .apply(
        lambda value:
        value if isinstance(value, list) else []
    )
)

print(
    "Final endpoint-artifact relationships:",
    len(endpoint_artifact_edges),
)

print(
    "Relationships with malicious evidence:",
    int(
        endpoint_artifact_edges["malicious_detected"].sum()
    ),
)

print(
    "Relationships without malicious evidence:",
    int(
        (~endpoint_artifact_edges["malicious_detected"]).sum()
    ),
)

Final endpoint-artifact relationships: 72
Relationships with malicious evidence: 60
Relationships without malicious evidence: 12


In [27]:
# Validate the final relationship table against the raw observation scope.
assert (
    endpoint_artifact_edges["record_count"].sum()
    == len(observation_scope)
), (
    "Relationship record counts do not account for all "
    "valid endpoint-artifact observations."
)

# Validate that each endpoint-artifact pair is unique.
assert not endpoint_artifact_edges.duplicated(
    subset=["endpoint_id", "sha256"]
).any(), (
    "Duplicate endpoint-artifact relationships detected."
)

# Verify that every malicious relationship has at least one detection.
assert (
    endpoint_artifact_edges.loc[
        endpoint_artifact_edges["malicious_detected"],
        "malicious_detection_count",
    ]
    > 0
).all(), (
    "A malicious relationship has zero malicious detections."
)

# Verify that every malicious security record maps to a final relationship.
malicious_keys = set(
    zip(
        malicious["endpoint_id"],
        malicious["sha256"],
    )
)

final_malicious_keys = set(
    zip(
        endpoint_artifact_edges.loc[
            endpoint_artifact_edges["malicious_detected"],
            "endpoint_id",
        ],
        endpoint_artifact_edges.loc[
            endpoint_artifact_edges["malicious_detected"],
            "sha256",
        ],
    )
)

assert malicious_keys.issubset(final_malicious_keys), (
    "Some malicious security records could not be mapped "
    "to endpoint-artifact relationships."
)

print("=" * 70)
print("FINAL RELATIONSHIP VALIDATION")
print("=" * 70)

print(
    "Raw valid observations:",
    len(observation_scope),
)

print(
    "Endpoint-artifact relationships:",
    len(endpoint_artifact_edges),
)

print(
    "Unique endpoints:",
    endpoint_artifact_edges["endpoint_id"].nunique(),
)

print(
    "Unique artifacts:",
    endpoint_artifact_edges["sha256"].nunique(),
)

print(
    "Relationships with malicious evidence:",
    int(
        endpoint_artifact_edges["malicious_detected"].sum()
    ),
)

print(
    "Relationships without malicious evidence:",
    int(
        (~endpoint_artifact_edges["malicious_detected"]).sum()
    ),
)

print("Validation passed.")

FINAL RELATIONSHIP VALIDATION
Raw valid observations: 157
Endpoint-artifact relationships: 72
Unique endpoints: 38
Unique artifacts: 60
Relationships with malicious evidence: 60
Relationships without malicious evidence: 12
Validation passed.


In [28]:
# Display a sample of relationships that contain explicit malicious evidence.
malicious_relationships = endpoint_artifact_edges[
    endpoint_artifact_edges["malicious_detected"]
].copy()

print("Sample malicious relationships:")
display(malicious_relationships.head(20))

Sample malicious relationships:


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count,malicious_detected,malicious_detection_count,malicious_file_names,first_malicious_detection,last_malicious_detection,first_quarantine_time,last_quarantine_time
0,EP2SJ6NE,93ffd792f92437a850e4939c84442391a624e20446cea2...,[libreoffice-calc.desktop],1,2026-09-26 11:06:14,2026-09-26 11:06:14,1,True,1,[libreoffice-calc.desktop],2026-09-26 11:06:14,2026-09-26 11:06:14,2026-09-25 11:34:00,2026-09-25 11:34:00
1,EP3GALVS,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,[recently-used.xbel],2,2026-09-19 16:35:19,2026-09-21 09:17:02,1,True,2,[recently-used.xbel],2026-09-19 16:35:19,2026-09-21 09:17:02,2026-09-19 13:18:00,2026-09-19 13:18:00
2,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[load-hub-i18n.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
3,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[shopping_iframe_driver.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
4,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[tokenized-card.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
5,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[wallet_checkout_autofill_driver.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
6,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[wallet-buynow.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
7,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[wallet_donation_driver.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
8,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[runtime.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
9,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,[load-ec-i18n.bundle.js],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00


In [29]:
# Export the complete endpoint-artifact relationship table.
# List-valued fields are converted into portable pipe-delimited strings.
edge_export = endpoint_artifact_edges.copy()

edge_export["file_names"] = (
    edge_export["file_names"]
    .apply(lambda values: " | ".join(values))
)

edge_export["malicious_file_names"] = (
    edge_export["malicious_file_names"]
    .apply(lambda values: " | ".join(values))
)

edge_export.to_csv(
    EDGE_OUTPUT_PATH,
    index=False,
)

print(
    f"Saved endpoint-artifact table: "
    f"{EDGE_OUTPUT_PATH.resolve()}"
)

Saved endpoint-artifact table: C:\Users\Asus\Desktop\web_activity\data\endpoint_artifact_edges.csv


In [30]:
# Reload the exported CSV to verify that the file written to disk
# contains the expected relationship-level schema.
exported_edges = pd.read_csv(EDGE_OUTPUT_PATH)

expected_columns = {
    "endpoint_id",
    "sha256",
    "file_names",
    "record_count",
    "first_event",
    "last_event",
    "file_name_count",
    "malicious_detected",
    "malicious_detection_count",
    "malicious_file_names",
    "first_malicious_detection",
    "last_malicious_detection",
    "first_quarantine_time",
    "last_quarantine_time",
}

missing_columns = expected_columns - set(exported_edges.columns)

if missing_columns:
    raise ValueError(
        f"Export is missing expected columns: {sorted(missing_columns)}"
    )

print("=" * 70)
print("EXPORTED FILE VERIFICATION")
print("=" * 70)

print("Rows:", len(exported_edges))
print("Columns:", len(exported_edges.columns))
print(
    "Malicious relationships:",
    int(
        exported_edges["malicious_detected"]
        .astype(str)
        .str.lower()
        .eq("true")
        .sum()
    ),
)

print("Export verification passed.")

display(exported_edges.head())

EXPORTED FILE VERIFICATION
Rows: 72
Columns: 14
Malicious relationships: 60
Export verification passed.


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count,malicious_detected,malicious_detection_count,malicious_file_names,first_malicious_detection,last_malicious_detection,first_quarantine_time,last_quarantine_time
0,EP2SJ6NE,93ffd792f92437a850e4939c84442391a624e20446cea2...,libreoffice-calc.desktop,1,2026-09-26 11:06:14,2026-09-26 11:06:14,1,True,1,libreoffice-calc.desktop,2026-09-26 11:06:14,2026-09-26 11:06:14,2026-09-25 11:34:00,2026-09-25 11:34:00
1,EP3GALVS,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,recently-used.xbel,2,2026-09-19 16:35:19,2026-09-21 09:17:02,1,True,2,recently-used.xbel,2026-09-19 16:35:19,2026-09-21 09:17:02,2026-09-19 13:18:00,2026-09-19 13:18:00
2,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,load-hub-i18n.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,load-hub-i18n.bundle.js,2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
3,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,shopping_iframe_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,shopping_iframe_driver.js,2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
4,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,tokenized-card.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,True,2,tokenized-card.bundle.js,2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00


In [31]:
# Compare the complete endpoint population from the raw telemetry with the
# endpoints that currently have at least one parseable SHA-256 artifact.
all_raw_endpoints = set(
    df["endpoint_id"]
    .dropna()
    .unique()
)

artifact_endpoints = set(
    endpoint_artifact_edges["endpoint_id"]
    .dropna()
    .unique()
)

# These endpoints exist in the raw telemetry but have no endpoint-artifact
# relationship because no usable SHA-256 was extracted for them.
endpoints_without_artifacts = sorted(
    all_raw_endpoints - artifact_endpoints
)

print("=" * 70)
print("ENDPOINT COVERAGE VALIDATION")
print("=" * 70)

print(
    "Total endpoints in raw telemetry:",
    len(all_raw_endpoints),
)

print(
    "Endpoints represented in endpoint-artifact table:",
    len(artifact_endpoints),
)

print(
    "Endpoints without a parseable artifact:",
    len(endpoints_without_artifacts),
)

print("\nEndpoints without parseable artifacts:")

for endpoint in endpoints_without_artifacts:
    print(endpoint)

ENDPOINT COVERAGE VALIDATION
Total endpoints in raw telemetry: 247
Endpoints represented in endpoint-artifact table: 38
Endpoints without a parseable artifact: 209

Endpoints without parseable artifacts:
EP2C4ZH5
EP2EVFKU
EP2YUIUX
EP3JZO7Y
EP3PJJZF
EP3SKSVQ
EP3XZALW
EP3Z5SXV
EP3ZPBUE
EP4C4STG
EP4HZ33W
EP4MZIBL
EP4PGHIT
EP4RS3P7
EP4WW2QO
EP55ZUGP
EP5DHOIP
EP5JYWEZ
EP5KBVLG
EP5NU2IP
EP5U5X2I
EP63QFAS
EP65VPYQ
EP67QOYP
EP6D7K7J
EP6HECD2
EP6IKT5R
EP6RMQC5
EP6SCJIM
EP6TZCQ6
EP6WDXYR
EP6WOB2Q
EP7FLWZB
EP7FZ2XY
EP7OWDTR
EP7PCCYI
EP7XJK3C
EPADG3JU
EPALDFJP
EPAM6DF7
EPAPPPPN
EPAR5TSH
EPARWEH7
EPAX5P5D
EPB5VWOI
EPB7PFDD
EPBG5SVE
EPBIBHNA
EPBIT7ZS
EPBMNHPG
EPBMNRN7
EPBUUL45
EPBVEPUN
EPBWRBCV
EPBZVWPM
EPC7TXDE
EPCKIU6N
EPCLJGKA
EPCPSGXR
EPCQBKSG
EPCRFN4D
EPD3I3JF
EPD7B7DQ
EPDO62BH
EPDR3POY
EPDVCZ3P
EPDYIGF3
EPE4RO5I
EPE556O2
EPE7YMMO
EPELOQ73
EPEPTYRN
EPEWYRFX
EPF5L3TA
EPFE66OS
EPFF7UHG
EPFQEJS4
EPFV5FDB
EPFYWUMZ
EPG52BAX
EPG67237
EPGBKH56
EPGHGYUU
EPGHTVEF
EPGJZAX6
EPGN44A5
EPGPX2E7
EPGQPYHM
EPH5

In [32]:
# Compare the endpoints with explicit malicious detections against the
# endpoints represented in the final endpoint-artifact relationship table.
# This verifies that malicious evidence is not being lost during parsing.

raw_malicious_endpoints = set(
    malicious["endpoint_id"]
    .dropna()
    .unique()
)

graph_malicious_endpoints = set(
    endpoint_artifact_edges.loc[
        endpoint_artifact_edges["malicious_detected"],
        "endpoint_id",
    ]
    .dropna()
    .unique()
)

# Identify malicious endpoints whose security evidence did not produce
# a corresponding endpoint-artifact relationship.
missing_malicious_endpoints = sorted(
    raw_malicious_endpoints - graph_malicious_endpoints
)

# Identify any graph malicious endpoints that are not present in the
# raw malicious endpoint population.
unexpected_malicious_endpoints = sorted(
    graph_malicious_endpoints - raw_malicious_endpoints
)

print("=" * 70)
print("MALICIOUS ENDPOINT CORRELATION VALIDATION")
print("=" * 70)

print(
    "Malicious endpoints in raw telemetry:",
    len(raw_malicious_endpoints),
)

print(
    "Malicious endpoints represented in artifact table:",
    len(graph_malicious_endpoints),
)

print(
    "Malicious endpoints missing from artifact table:",
    len(missing_malicious_endpoints),
)

print(
    "Unexpected malicious endpoints:",
    len(unexpected_malicious_endpoints),
)

print("\nMissing malicious endpoints:")

for endpoint in missing_malicious_endpoints:
    print(endpoint)

print("\nUnexpected malicious endpoints:")

for endpoint in unexpected_malicious_endpoints:
    print(endpoint)

MALICIOUS ENDPOINT CORRELATION VALIDATION
Malicious endpoints in raw telemetry: 33
Malicious endpoints represented in artifact table: 33
Malicious endpoints missing from artifact table: 0
Unexpected malicious endpoints: 0

Missing malicious endpoints:

Unexpected malicious endpoints:
